In [ ]:
import math
import re
import drawsvg as draw
import numpy as np

# =====================================
# Inputs (all lengths in mm)
# =====================================
# --- geometry ---
width = 19.5            # footprint width (given)
edge_offset = 0.6       # keep-out from the footprint edge
trace_width = 0.15      # W: the trace "gauge". Change it and everything below re-solves.
trace_spacing = trace_width
corner_style = "45"     # "45": straight runs joined by 45-degree chamfers; "arc": rounded quarter-circle bends
chamfer = None          # "45" mode only: size of each 45-degree bend (mm, along each axis); None -> trace_width / 2
min_run = 1.0           # shortest allowed vertical run of the serpentine
height_round = 0.01     # solved height is rounded UP to this step
tails_at_top = True     # True: both tails exit the top edge; False: the bottom edge
draw_outline = True     # draw the single width x height rectangle
image_scale = 15        # preview only
kicad_margin = 1.0      # blank border (mm) around the footprint in the KiCad SVGs

# --- electrical (target applies to the ONE full-width trace) ---
target_resistance = 6.0     # ohms, at T_amb
copper_thickness = 0.035    # T, 1 oz/ft^2 copper
resistivity = 1.72e-8       # rho, ohm*m at T_ref (copper)
alpha = 0.00393             # temperature coefficient of resistivity, 1/degC (copper)
T_ref = 25.0                # reference temperature in the formula, degC
T_amb = 25.0                # ambient temperature, degC

# =====================================

assert width > 0
assert edge_offset >= 0 and edge_offset < width / 2
assert target_resistance > 0 and copper_thickness > 0 and trace_width > 0
assert corner_style in ("45", "arc"), 'corner_style must be "45" or "arc"'

pitch = trace_width + trace_spacing                  # centre-to-centre distance of adjacent runs
if chamfer is None:
    chamfer = trace_width / 2
# corner size k: arcs use radius trace_width/2, 45-degree bends use the chamfer
corner = trace_width / 2 if corner_style == "arc" else chamfer
assert 0 < corner and 2 * corner < pitch - 1e-9, \
    f"bend size {corner} too big: needs 2*bend < run pitch {pitch} so a flat piece remains between the two bends"

RHO_OHM_MM = resistivity * 1000.0  # ohm*m -> ohm*mm so everything stays in mm


def temp_factor(t_amb=None):
    t_amb = T_amb if t_amb is None else t_amb
    return 1 + alpha * (t_amb - T_ref)


def trace_resistance(length_mm, t_amb=None):
    """R = rho * L / (T * W) * (1 + alpha * (T_amb - 25))"""
    return RHO_OHM_MM * length_mm / (copper_thickness * trace_width) * temp_factor(t_amb)


def length_for_resistance(r_ohm, t_amb=None):
    """Inverse of trace_resistance: the trace length that gives r_ohm."""
    return r_ohm * copper_thickness * trace_width / (RHO_OHM_MM * temp_factor(t_amb))


def path_length(dstr):
    """Length of an M/L/A path string (arcs are quarter circles)."""
    segs = dstr.split(" ")
    ops = [x[0] for x in segs]
    assert ops[0] == "M"
    assert "M" not in ops[1:]

    total_len = 0
    origin = [float(x) for x in segs[0][1:].split(",")]
    for seg in segs[1:]:
        nex = [float(x) for x in seg[1:].split(",")]
        if seg[0] == "L":
            total_len += np.sqrt((nex[0] - origin[0]) ** 2 + (nex[1] - origin[1]) ** 2)
            origin = nex
        elif seg[0] == "A":
            assert nex[0] == nex[1], "Not a circle"
            assert np.abs(np.abs(nex[5] - origin[0]) - nex[0]) < 0.001, "Bad arclen"
            assert np.abs(np.abs(nex[6] - origin[1]) - nex[1]) < 0.001, "Bad arclen"
            total_len += np.pi * nex[0] / 2
            origin = [nex[5], nex[6]]
        else:
            assert False, seg[0]
    return total_len


def calc_length(obj):
    return path_length(vars(obj)["args"]["d"])


# ---------------------------------------------------------------
# Column layout: ONE serpentine centred across the full width
# ---------------------------------------------------------------
long_width = trace_width * 2 + trace_spacing * 2     # one up-run + one down-run
x_margin = 2 * edge_offset                           # same side margin as before
n_cols = int(math.floor((width - 2 * x_margin + trace_spacing) / long_width + 1e-9))
assert n_cols >= 1, "width too small for even one serpentine column"
span = n_cols * long_width - trace_spacing           # left edge of metal -> right edge of metal
x0 = (width - span) / 2                              # centres the trace in the footprint
x_boxes = x0 + long_width * np.arange(n_cols)


# ---------------------------------------------------------------
# Trace geometry as a segment list (tails at the bottom, y < 0),
# then turned into an SVG path, mirrored in y if tails_at_top.
# ---------------------------------------------------------------
def trace_segments(h):
    tw, ts, r, k = trace_width, trace_spacing, trace_width / 2, corner
    top = h - edge_offset                 # usable far edge (outer edge of the metal)
    top_flat, top_end = top - r, top - r - k   # centre line of the far-end flat / where the runs stop below it
    bot_flat, bot_start = -r, -r + k           # same at the tail end

    def bend(x, y, sweep):
        """A rounded bend (arc) or a 45-degree bend (straight diagonal) ending at (x, y)."""
        return ("A", x, y, sweep) if corner_style == "arc" else ("L", x, y)

    segs = [("M", x0 + r, -2 * tw), ("L", x0 + r, 0)]
    for i, l in enumerate(x_boxes):
        xa = l + r                        # up-run centre
        xb = xa + pitch                   # down-run centre
        segs += [
            ("L", xa, top_end),
            bend(xa + k, top_flat, False),
            ("L", xb - k, top_flat),
            bend(xb, top_end, False),
            ("L", xb, bot_start),
        ]
        if i != len(x_boxes) - 1:
            xn = xa + long_width          # next up-run centre
            segs += [
                bend(xb + k, bot_flat, True),
                ("L", xn - k, bot_flat),
                bend(xn, bot_start, True),
            ]
    segs.append(("L", x_boxes[-1] + r + pitch, -2 * tw))
    return segs


def to_path(segs, h, flip, color="green"):
    p = draw.Path(stroke_width=trace_width, stroke=color, fill_opacity=0)
    for s in segs:
        op, x, y = s[0], s[1], s[2]
        y = h - y if flip else y
        if op == "M":
            p.M(x, y)
        elif op == "L":
            p.L(x, y)
        else:
            sweep = (not s[3]) if flip else s[3]  # mirroring reverses arc direction
            p.A(trace_width / 2, trace_width / 2, 0, False, sweep, x, y)
    return p


def build_trace(h):
    return to_path(trace_segments(h), h, tails_at_top)


# ---------------------------------------------------------------
# Solve for the footprint height that gives the target resistance
# ---------------------------------------------------------------
# Trace length is linear in height (every run grows 1:1 with it), so two
# evaluations are enough to solve exactly.
height_min = edge_offset + trace_width + min_run
len_min = calc_length(build_trace(height_min))
slope = (calc_length(build_trace(height_min + 10)) - len_min) / 10  # mm of trace per mm of height

len_target = length_for_resistance(target_resistance)
r_min = trace_resistance(len_min)
assert len_target >= len_min, (
    f"Target {target_resistance:.3f} ohm is below the smallest resistance this width can give "
    f"({r_min:.3f} ohm at height {height_min:.2f} mm). Increase the target, narrow the trace "
    f"(or thin the copper), or reduce width/min_run."
)

height_exact = height_min + (len_target - len_min) / slope
height = round(math.ceil(height_exact / height_round - 1e-9) * height_round, 6)

# ---------------------------------------------------------------
# Final geometry and report
# ---------------------------------------------------------------
path_trace = build_trace(height)
length_trace = calc_length(path_trace)
R_actual = trace_resistance(length_trace)

outline = draw.Path(stroke_width=trace_width, stroke="RED", fill_opacity=0)
outline.M(0, 0)
outline.L(width, 0)
outline.L(width, height)
outline.L(0, height)
outline.Z()

print(f"footprint        : {width:.2f} x {height:.2f} mm (width given, height solved)")
print(f"target           : {target_resistance:.4f} ohm at {T_amb:.0f} degC -> length {len_target:.2f} mm")
print(f"solved height    : {height_exact:.3f} mm exact, {height:.2f} mm rounded up")
print(f"serpentine       : {n_cols} columns, tails at the {'top' if tails_at_top else 'bottom'}")
print(f"trace gauge      : {trace_width} mm wide, {trace_spacing} mm spacing, bends: "
      + (f"45-degree chamfers ({corner} mm)" if corner_style == "45" else "rounded arcs"))
print(f"trace length     : {length_trace:.2f} mm")
print(f"trace resistance : {R_actual:.4f} ohm ({(R_actual / target_resistance - 1) * 100:+.3f}% vs target)")
print(f"min possible     : {r_min:.3f} ohm at height {height_min:.2f} mm")

# ---------------------------------------------------------------
# Preview
# ---------------------------------------------------------------
d = draw.Drawing(width + 20, height + 20, origin=(-10, -10), displayInline=False)
if draw_outline:
    d.draw(outline)
d.draw(path_trace)
d.save_svg('curve_verbose.svg')
d.rasterize()
d.set_pixel_scale(image_scale)
d


In [ ]:
# ---------------------------------------------------------------
# KiCad export: SVGs whose width/height are real millimetres
# ---------------------------------------------------------------
def save_svg_mm(drawing, filename, w_mm, h_mm):
    """Save an SVG whose width/height are in mm, so 1 viewBox unit = 1 mm."""
    drawing.save_svg(filename)
    s = open(filename).read()
    tag = re.search(r"<svg[^>]*>", s).group(0)
    assert "viewBox" in tag, "no viewBox found"
    new = re.sub(r'(?<![\w-])width="[^"]*"', f'width="{w_mm:.4f}mm"', tag, count=1)
    new = re.sub(r'(?<![\w-])height="[^"]*"', f'height="{h_mm:.4f}mm"', new, count=1)
    open(filename, "w").write(s.replace(tag, new, 1))


def kicad_drawing(item):
    m = kicad_margin
    k = draw.Drawing(width + 2 * m, height + 2 * m, origin=(-m, -m), displayInline=False)
    k.draw(item)
    return k


# Both files share the same canvas size, so they line up when imported.
canvas_w, canvas_h = width + 2 * kicad_margin, height + 2 * kicad_margin
save_svg_mm(kicad_drawing(path_trace), "traces_kicad.svg", canvas_w, canvas_h)
save_svg_mm(kicad_drawing(outline), "outline_kicad.svg", canvas_w, canvas_h)
print(f"wrote traces_kicad.svg and outline_kicad.svg ({canvas_w:.2f} x {canvas_h:.2f} mm canvas)")


In [ ]:
# Check cell: reads the exported SVGs back from disk and verifies them.
import xml.etree.ElementTree as ET


def read_svg(fn):
    root = ET.parse(fn).getroot()
    paths = [el.attrib["d"] for el in root.iter() if el.tag.endswith("path")]
    w_attr, h_attr = root.attrib["width"], root.attrib["height"]
    assert w_attr.endswith("mm") and h_attr.endswith("mm"), f"{fn}: width/height are not in mm"
    vb = [float(v) for v in re.split(r"[,\s]+", root.attrib["viewBox"].strip())]
    assert abs(float(w_attr[:-2]) - vb[2]) < 1e-3 and abs(float(h_attr[:-2]) - vb[3]) < 1e-3, \
        f"{fn}: mm size does not match viewBox (scale is not 1 unit = 1 mm)"
    return paths, (float(w_attr[:-2]), float(h_attr[:-2]))


def endpoints(dstr):
    """End point of every M/L/A segment."""
    pts = []
    for seg in dstr.split(" "):
        if seg[0] not in "MLA":
            continue
        nums = [float(x) for x in re.split(r"[,\s]+", seg[1:]) if x]
        pts.append((nums[-2], nums[-1]))
    return pts


# ---- traces ----
paths, (w_mm, h_mm) = read_svg("traces_kicad.svg")
assert len(paths) == 1, f"expected 1 trace path, got {len(paths)}"
assert abs(w_mm - (width + 2 * kicad_margin)) < 1e-3 and abs(h_mm - (height + 2 * kicad_margin)) < 1e-3
pts = endpoints(paths[0])

tail_y = height + 2 * trace_width if tails_at_top else -2 * trace_width
assert abs(pts[0][1] - tail_y) < 1e-9 and abs(pts[-1][1] - tail_y) < 1e-9, "tails are not on the expected edge"
assert pts[0][0] == pts[1][0] and pts[-1][0] == pts[-2][0], "slanted tail segment"

lo = edge_offset if tails_at_top else -2 * trace_width
hi = height + 2 * trace_width if tails_at_top else height - edge_offset
for x, y in pts:
    assert edge_offset - 1e-9 <= x <= width - edge_offset + 1e-9, f"x={x} outside keep-out"
    assert lo - 1e-9 <= y <= hi + 1e-9, f"y={y} outside allowed band [{lo}, {hi}]"

if corner_style == "45":
    assert "A" not in [seg[0] for seg in paths[0].split(" ")], "arc found in a 45-degree trace"
    for (x1, y1), (x2, y2) in zip(pts, pts[1:]):
        dx, dy = abs(x2 - x1), abs(y2 - y1)
        assert dx < 1e-9 or dy < 1e-9 or abs(dx - dy) < 1e-9, \
            f"segment ({x1:.3f},{y1:.3f})->({x2:.3f},{y2:.3f}) is not a multiple of 45 degrees"

xs = [x for x, _ in pts]
assert abs((min(xs) + max(xs)) - width) < 1e-6, "trace is not centred across the width"

r = trace_resistance(path_length(paths[0]))
assert target_resistance - 1e-9 <= r <= target_resistance * 1.01, \
    f"R={r:.4f} ohm is not within 0..+1% of target {target_resistance} ohm"
print(f"trace: length {path_length(paths[0]):.2f} mm -> {r:.4f} ohm")

# ---- outline ----
paths_o, (w_o, h_o) = read_svg("outline_kicad.svg")
assert len(paths_o) == 1, f"expected 1 outline path, got {len(paths_o)}"
assert (w_o, h_o) == (w_mm, h_mm), "outline and traces canvases differ, they won't line up"
opts = endpoints(paths_o[0])
ox, oy = [p[0] for p in opts], [p[1] for p in opts]
assert abs(min(ox)) < 1e-9 and abs(max(ox) - width) < 1e-9, "outline width is wrong"
assert abs(min(oy)) < 1e-9 and abs(max(oy) - height) < 1e-9, "outline height is wrong"
print(f"outline: {max(ox) - min(ox):.2f} x {max(oy) - min(oy):.2f} mm")

print("KiCad SVGs OK")
